Step 1 — Data Preprocessing & Understanding
ModelX Optimization Sprint: dementia risk prediction from non-medical features

This notebook:
1. Loads the dataset and the variable classification table (derived from the NACC data dictionary)
2. Confirms which columns exist in the actual CSV vs. the classification table
3. Profiles missingness, dtypes, and sentinel/special codes (-4, 8/88/888, 9/99/999, etc.)
4. Checks the dementia label and class balance
5. Produces the filtered "non-medical only" feature set for later steps

In [1]:
import pandas as pd
import numpy as np
import os
 
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

In [2]:
DATA_PATH    = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\Dementia_Prediction_Dataset.csv"  # <-- your CSV
OUTPUT_DIR   = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs"
LABEL_COL    = "NACCUDSD"   # 1=Normal, 2=Impaired-not-MCI, 3=MCI, 4=Dementia
ID_COL       = "NACCID"     # subject identifier
 
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
FEATURE_COLS = [
    # --- Demographics (Form A1) ---(11)
    "BIRTHMO",    # birth month
    "BIRTHYR",    # birth year
    "NACCAGEB",   # derived baseline age at initial visit
    "SEX",        # sex
    "HISPANIC",   # hispanic/latino ethnicity
    "HISPOR",     # hispanic origin subgroup
    "RACE",       # primary race
    "RACESEC",    # secondary race
    "RACETER",    # tertiary race
    "PRIMLANG",   # primary language
    "HANDED",     # handedness
    # --- Socioeconomic / Social (Form A1) ---(4)
    "EDUC",       # years of education
    "MARISTAT",   # marital status
    "NACCLIVS",   # living situation
    "RESIDENC",   # type of residence
    # --- Lifestyle (Form A5) ---(7)
    "TOBAC30",    # smoked in last 30 days
    "TOBAC100",   # smoked 100+ cigarettes lifetime
    "SMOKYRS",    # total years smoked
    "PACKSPER",   # average packs per day
    "QUITSMOK",   # age quit smoking
    "ALCOCCAS",   # any alcohol in last 3 months
    "ALCFREQ",    # frequency of alcohol consumption
    # --- Social engagement / Co-participant (Form A2) ---(5)
    "INRELTO",    # co-participant relationship to subject
    "INKNOWN",    # how long co-participant has known subject
    "INLIVWTH",   # co-participant lives with subject
    "INVISITS",   # frequency of in-person visits
    "INCALLS",    # frequency of telephone contact
]

In [4]:
# 3. Load only the needed columns
# First peek at header only to catch naming mismatches before the full read
header_df = pd.read_csv(DATA_PATH, nrows=0)
actual_cols = set(header_df.columns)
 
wanted_cols = set(FEATURE_COLS) | {LABEL_COL, ID_COL}
missing_from_data = sorted(c for c in wanted_cols if c not in actual_cols)
if missing_from_data:
    print(f"WARNING: {len(missing_from_data)} wanted columns NOT found in the CSV header:")
    print(missing_from_data)
else:
    print("All wanted columns found in the CSV header.")
 
usecols = [c for c in [ID_COL, LABEL_COL] + FEATURE_COLS if c in actual_cols]
df = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)
 
print(f"\nDataset shape (filtered): {df.shape}")
print(f"Feature columns loaded: {len([c for c in FEATURE_COLS if c in df.columns])}/{len(FEATURE_COLS)}")

All wanted columns found in the CSV header.

Dataset shape (filtered): (195196, 29)
Feature columns loaded: 27/27


In [5]:
# 4. Profile: dtypes, missingness, cardinality

feature_cols_in_df = [c for c in FEATURE_COLS if c in df.columns]
 
profile = pd.DataFrame({
    "dtype":      df[feature_cols_in_df].dtypes,
    "n_missing":  df[feature_cols_in_df].isna().sum(),
    "pct_missing":(df[feature_cols_in_df].isna().mean() * 100).round(1),
    "n_unique":   df[feature_cols_in_df].nunique(),
    "sample_vals":df[feature_cols_in_df].apply(lambda s: s.dropna().unique()[:5].tolist()),
})
profile = profile.sort_values("pct_missing", ascending=False)
print(profile.to_string())

          dtype  n_missing  pct_missing  n_unique                     sample_vals
BIRTHMO   int64          0          0.0        12                [5, 12, 1, 2, 7]
BIRTHYR   int64          0          0.0       106  [1952, 1956, 1958, 1945, 1942]
NACCAGEB  int64          0          0.0        90            [70, 66, 63, 77, 81]
SEX       int64          0          0.0         2                          [1, 2]
HISPANIC  int64          0          0.0         3                       [0, 1, 9]
HISPOR    int64          0          0.0         9              [88, 1, 99, 2, 50]
RACE      int64          0          0.0         7               [1, 99, 3, 50, 5]
RACESEC   int64          0          0.0         8              [88, 3, 50, 99, 4]
RACETER   int64          0          0.0         8              [88, 50, 99, 2, 1]
PRIMLANG  int64          0          0.0         8                 [1, 2, 8, 3, 6]
HANDED    int64          0          0.0         4                    [2, 1, 3, 9]
EDUC      int64 

In [6]:
# 5. Sentinel / special missing code check
# NACC UDS uses numeric sentinel codes that look like real values to pandas:
#   -4  = not applicable (form version didn't collect this)
#    9  = unknown
#   99  = unknown
#   88  = not applicable / not assessed
#  888  = unknown or not assessed
# These must be recoded to NaN before modeling.

SENTINEL_CANDIDATES = [-4, 8, 9, 88, 99, 888, 999, 8888, 9999]
 
print("Sentinel code scan (cross-check each against data dictionary before recoding):\n")
for col in feature_cols_in_df:
    if pd.api.types.is_numeric_dtype(df[col]):
        vc = df[col].value_counts(dropna=False)
        hits = {s: int(vc.get(s, 0)) for s in SENTINEL_CANDIDATES if vc.get(s, 0) > 0}
        if hits:
            print(f"  {col}: {hits}")

Sentinel code scan (cross-check each against data dictionary before recoding):

  BIRTHMO: {8: 17600, 9: 16676}
  NACCAGEB: {88: 1610, 99: 26}
  HISPANIC: {9: 677}
  HISPOR: {88: 181506, 99: 399}
  RACE: {99: 832}
  RACESEC: {88: 188382, 99: 1559}
  RACETER: {88: 192713, 99: 1372}
  PRIMLANG: {8: 2756, 9: 276}
  HANDED: {9: 1032}
  EDUC: {8: 1777, 9: 1118, 99: 974}
  MARISTAT: {9: 896}
  NACCLIVS: {9: 324}
  RESIDENC: {9: 2878}
  TOBAC30: {-4: 71664, 9: 496}
  TOBAC100: {-4: 71664, 9: 1260}
  SMOKYRS: {-4: 71664, 8: 725, 9: 393, 88: 422, 99: 3476}
  PACKSPER: {-4: 71664, 8: 1153, 9: 2649}
  QUITSMOK: {-4: 71664, 8: 2, 9: 3, 88: 12, 99: 1, 888: 71373, 999: 3623}
  ALCOCCAS: {-4: 175552, 9: 318}
  ALCFREQ: {-4: 175870, 8: 7014, 9: 71}
  INRELTO: {-4: 8288}
  INKNOWN: {-4: 171430, 8: 185, 9: 117, 88: 3, 99: 44, 999: 1050}
  INLIVWTH: {-4: 8288}
  INVISITS: {-4: 8288, 8: 112103}
  INCALLS: {-4: 8288, 8: 112103}


In [7]:
# 6. Define binary label
#
# NACCUDSD: 1=Normal, 2=Impaired-not-MCI, 3=MCI, 4=Dementia
#
# Three options (document your choice in the report):
#   A (strict):        dementia=1 iff NACCUDSD==4
#   B (at-risk):       dementia=1 iff NACCUDSD in {3,4}   <- matches "risk of dementia" scenario framing
#   C (clean split):   dementia=1 iff NACCUDSD==4, dementia=0 iff NACCUDSD==1, drop rows 2 & 3

LABEL_DEFINITION = "B"   # <-- change to "A" or "C" if preferred; justify in report
 
print(f"NACCUDSD raw distribution:\n{df[LABEL_COL].value_counts(dropna=False).sort_index()}\n")
 
if LABEL_DEFINITION == "A":
    df["dementia_binary"] = (df[LABEL_COL] == 4).astype(int)
elif LABEL_DEFINITION == "B":
    df["dementia_binary"] = df[LABEL_COL].isin([3, 4]).astype(int)
elif LABEL_DEFINITION == "C":
    df = df[df[LABEL_COL].isin([1, 4])].copy()
    df["dementia_binary"] = (df[LABEL_COL] == 4).astype(int)
else:
    raise ValueError("LABEL_DEFINITION must be 'A', 'B', or 'C'")
 
balance = (df["dementia_binary"].value_counts(normalize=True) * 100).round(1)
print(f"Rows after label definition '{LABEL_DEFINITION}': {len(df)}")
print(f"Class balance (%):\n{balance}")

NACCUDSD raw distribution:
NACCUDSD
1    94933
2     8567
3    34106
4    57590
Name: count, dtype: int64

Rows after label definition 'B': 195196
Class balance (%):
dementia_binary
0    53.0
1    47.0
Name: proportion, dtype: float64


In [8]:
# 7. Subject-level visit structure

visits_per_subject = df.groupby(ID_COL).size()
print(f"\nVisit count per subject:\n{visits_per_subject.describe().round(2)}")
print(f"\nSubjects with >1 visit: {(visits_per_subject > 1).sum()} / {len(visits_per_subject)}")
 
# DECISION (justify in report — choose one):
#   (a) Baseline only: keep only the first visit per subject (lowest leakage risk)
#   (b) All visits:    keep all rows but split train/test by subject ID to prevent leakage
#   (c) Aggregated:    collapse multiple visits into one row per subject


Visit count per subject:
count    52537.00
mean         3.72
std          3.30
min          1.00
25%          1.00
50%          3.00
75%          5.00
max         20.00
dtype: float64

Subjects with >1 visit: 35671 / 52537


In [9]:
# 8. Save for Step 2

out_data    = os.path.join(OUTPUT_DIR, "step1_filtered_dataset.csv")
out_profile = os.path.join(OUTPUT_DIR, "step1_feature_profile.csv")
 
df[[ID_COL, "dementia_binary"] + feature_cols_in_df].to_csv(out_data, index=False)
profile.to_csv(out_profile)
print(f"Saved:\n  {out_data}\n  {out_profile}")

Saved:
  C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step1_filtered_dataset.csv
  C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step1_feature_profile.csv
